# Clase 159 — Modelos de difusión (DDPM, score-based)

Los **modelos de difusión** (Ho et al., 2020) destronaron a los GANs en 2022. Un
**forward process** agrega ruido gaussiano gradualmente; un **reverse process**
aprendido lo elimina paso a paso. La schedule (`β_t`, `ᾱ_t`) y el forward
`q(x_t|x_0)` son **numpy puro y ejecutable**; la U-Net va conceptual.

**Requiere:** numpy (ejecutable). Para la U-Net real, `tensorflow`/`keras`.

## 1. Entorno + señal de juguete

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
except Exception as e:
    HAS_TF = False

# x_0: una senal 1D limpia (haria las veces de imagen)
x0 = np.sin(np.linspace(0, 4 * np.pi, 64)).astype('float32')
print('x0 shape:', x0.shape)

## 2. Noise schedule: `β_t`, `α_t`, `ᾱ_t` (ejecutable)

Schedule lineal de betas. `α_t = 1 - β_t`; `ᾱ_t = ∏ α_s` (cumulative product).

In [ ]:
T = 200
betas = np.linspace(1e-4, 0.02, T).astype('float32')       # schedule lineal
alphas = 1.0 - betas
alphas_cumprod = np.cumprod(alphas)                         # alpha_barra_t
print('beta_0 =', betas[0], '| beta_T =', betas[-1])
print('alpha_bar_0 =', round(float(alphas_cumprod[0]), 4),
      '| alpha_bar_T =', round(float(alphas_cumprod[-1]), 6))

## 3. Forward `q(x_t | x_0)` en forma cerrada (ejecutable)

`x_t = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε`, con `ε ~ N(0, I)`. Permite saltar a cualquier
`t` sin iterar.

In [ ]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = np.random.normal(size=x0.shape).astype('float32')
    sqrt_ab = np.sqrt(alphas_cumprod[t])
    sqrt_om = np.sqrt(1.0 - alphas_cumprod[t])
    return sqrt_ab * x0 + sqrt_om * noise, noise

for t in (0, 50, 150, 199):
    x_t, _ = q_sample(x0, t)
    print(f't={t:3d}  std(x_t)={x_t.std():.3f}  (mas ruido al crecer t)')

## 4. Objetivo de entrenamiento: predecir el ruido `ε_θ(x_t, t)`

Loss simple de Ho (2020): `MSE(ε, ε_θ(x_t, t))`. La red aprende a estimar el ruido
que se agregó. Acá un `predict_noise` placeholder (la U-Net real lo reemplaza).

In [ ]:
def predict_noise(x_t, t):
    # Placeholder conceptual: la U-Net entrenada devolveria eps_theta(x_t, t).
    # Sin entrenar, aproximamos con ceros para ilustrar el flujo del sampling.
    return np.zeros_like(x_t)

# En entrenamiento real:  loss = mean((eps - predict_noise(x_t, t))**2)
x_t, eps = q_sample(x0, 100)
loss = np.mean((eps - predict_noise(x_t, 100)) ** 2)
print('MSE(eps, eps_theta) del placeholder:', round(float(loss), 4))

## 5. U-Net como denoiser (conceptual, Keras)

En la práctica `ε_θ` es una **U-Net** (encoder-decoder con skip connections) que
recibe `x_t` y un embedding del timestep `t`.

In [ ]:
if HAS_TF:
    def tiny_unet(dim=64):
        x_in = keras.Input(shape=(dim, 1))
        t_in = keras.Input(shape=(1,))                     # timestep embebido
        h = layers.Conv1D(32, 3, padding='same', activation='swish')(x_in)
        skip = h
        h = layers.Conv1D(64, 3, padding='same', activation='swish')(h)
        h = layers.Conv1D(32, 3, padding='same', activation='swish')(h)
        h = layers.Add()([h, skip])                        # skip connection
        out = layers.Conv1D(1, 3, padding='same')(h)       # predice el ruido
        return keras.Model([x_in, t_in], out, name='tiny_unet')
    unet = tiny_unet()
    print('U-Net (skip connections) params:', unet.count_params())
else:
    print('U-Net: encoder-decoder con skip connections; entrada (x_t, t); salida eps.')

## 6. Reverse sampling DDPM (esqueleto ejecutable)

Desde `x_T ~ N(0,I)` se denoise paso a paso:
`x_{t-1} = 1/√α_t · (x_t - β_t/√(1-ᾱ_t) · ε_θ) + √β_t · z`.

In [ ]:
def ddpm_sample(shape, steps=T):
    x = np.random.normal(size=shape).astype('float32')     # x_T
    for t in reversed(range(steps)):
        eps = predict_noise(x, t)
        coef = betas[t] / np.sqrt(1.0 - alphas_cumprod[t])
        mean = (x - coef * eps) / np.sqrt(alphas[t])
        z = np.random.normal(size=shape).astype('float32') if t > 0 else 0.0
        x = mean + np.sqrt(betas[t]) * z
    return x

sample = ddpm_sample(x0.shape)
print('muestra generada shape:', sample.shape)
# DDIM / DPM-Solver reducen los 1000 -> 20-50 pasos manteniendo calidad.

## 7. Score-based (nota)

La formulación **score-based** (Song & Ermon) modela `∇_x log p(x)` (el *score*) y
generaliza DDPM vía SDEs. Predecir el ruido `ε_θ` equivale (salvo escala) a
estimar el score. **Latent diffusion** (Stable Diffusion) aplica todo esto en el
espacio comprimido de un VAE → ~8× más rápido.

## Ejercicios

1. **Schedule**: cambiá la schedule lineal por **cosine** y compar(¿ᾱ_t decae más suave?).
2. **Forward**: graficá `x_t` para `t ∈ {0, 25, 50, 100, 199}` y observá la corrupción.
3. **DDPM en MNIST**: reemplazá el placeholder por la U-Net entrenada y sampleá 64 imágenes.
4. **DDIM**: implementá el sampler determinista y reducí a 20-50 pasos.
5. **CFG**: con un modelo condicional, variá `guidance_scale ∈ {1, 5, 15}`.

## Conclusiones

- El forward `q(x_t|x_0) = √ᾱ_t·x_0 + √(1-ᾱ_t)·ε` tiene forma cerrada (salto directo a `t`).
- La schedule (`β_t`, `ᾱ_t`) controla cuánto ruido se agrega en cada paso.
- Se entrena una U-Net para **predecir el ruido** con `MSE(ε, ε_θ)`.
- El reverse sampling denoise paso a paso desde `N(0,I)`; DDIM/DPM-Solver lo aceleran.
- Score-based generaliza DDPM; latent diffusion lo hace práctico (Stable Diffusion).